# E231 — Federated and Scalable Product Rating Prediction: project run (reported results)

One notebook runs the whole project on Google Cloud Dataproc with **`config.yaml`** (Electronics, Books, Home_and_Kitchen, ~94M ratings). It shows the data engineering at scale with charts at each stage:

| § | Stage | Code that runs (unchanged team code) | Charts |
|---|---|---|---|
| 1–5 | Settings, setup, Google Cloud + quota check, raw data (parallel), optional tests | | |
| 6 | One cluster for the whole session, with a Kafka broker on the master | `infra/dataproc_launch.sh` (`KAFKA=1`), `infra/kafka_init.sh` | |
| 7 | **Module 1 — ETL** | `scripts/run_etl.py` | |
| 8 | Data profile | `scripts/run_profile.py` (read-only) | ETL funnel, per category, distributions |
| 9 | **Module 2 — centralized model** | `scripts/run_centralized.py` | accuracy, cold/warm, per category |
| 10 | **Module 3 — simulated FedAvg** | `scripts/run_federated.py` | convergence, cold/warm, communication |
| 11 | **Module 4 — scaling benchmarks** | `scripts/run_benchmarks.py` | strong + weak scaling (Amdahl / Gustafson), verdicts |
| 12 | **Module 4 — Kafka streaming** + live view | `scripts/run_streaming.py` (Navneet's `streaming/`) | live batches |
| 13 | **Streaming dashboard** | `nbtools/dashboard.py` | KPI tiles, latency, throughput |
| 14–15 | Save everything to Drive, **delete the cluster** | | |

**Demo mode:** with `SKIP_DONE = True`, every stage whose results are already in the bucket is skipped, and a cluster is only created when a stage needs one, so a rerun during a presentation only redraws the charts.

⚠️ A running cluster is billed. If a cell fails while a cluster is up, run **§15 Delete the cluster**. FedAvg is simulated on public, already-centralized data: no privacy guarantee is provided or implied.

## 1. Settings (the only cell you need to edit)

In [ ]:
PROJECT_ID = "your-gcp-project-id"            # e.g. "my-project-123456"
BUCKET     = "your-unique-bucket-name"        # no gs:// prefix, e.g. "my-e231-bucket"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/E231project"   # the uploaded folder (one nested level is found automatically)

WORKERS   = 16        # config.yaml's sweeps go up to 16 executors (1 per worker). Needs ~68 vCPUs and 1,700 GB SSD of quota.
SKIP_DONE = True      # skip stages whose outputs already exist in the bucket (rerun = charts only)

RAW_SOURCE    = "ucsd"   # "ucsd": stream CSVs from the UCSD site into the bucket | "drive": copy from DRIVE_RAW_DIR
DRIVE_RAW_DIR = "/content/drive/MyDrive/amazon_reviews_2018/ratings_only"

RUN_LOCAL_TESTS = False  # ~2 min in Colab, no cluster: FedAvg must match the NumPy reference (ALL PASSED)
RUN_TUNING      = False  # one-time rank/reg_param grid (als_tuning.json); you then copy the winners into config.yaml
FED_CLIENTS     = None   # None = all of federated.num_clients ([1,4,8,16]); [8] = only the saved federated model
RUN_BENCHMARKS  = True   # strong + weak scaling sweeps (scaling.* in config.yaml: 3 repetitions -> long)
BENCH_MODES     = None   # None = centralized + federated (+ etl if scaling.etl_sweep_enabled); e.g. ["centralized"]
BENCH_EXECUTORS = None   # None = all of scaling.executor_counts; e.g. [2, 4] for a partial sweep
BENCH_REPETITIONS = None # None = scaling.repetitions

RUN_STREAMING   = True   # Kafka on the master, scorer + producer jobs, summary, dashboard
STREAM_MODEL    = "centralized"   # or "federated" (simulated on-device scoring)
STREAM_WARMUP_SEC = 120  # scorer start-up (loads the model and caches serving tables) before the producer starts
RUN_RESTART_CHECK = True # rerun the scorer on the same checkpoint: expect no new batches and 0 duplicates

## 2. Sign in, copy the project, fill in project ID and bucket

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import os, re, sys, json, copy, time, yaml, subprocess
assert PROJECT_ID != "your-gcp-project-id", "Set PROJECT_ID in the Settings cell."
assert BUCKET != "your-unique-bucket-name", "Set BUCKET in the Settings cell (names are global: pick your own)."
assert re.fullmatch(r"[a-z0-9][a-z0-9._-]{1,61}[a-z0-9]", BUCKET), "BUCKET: lowercase letters, digits, - _ . ; 3-63 chars"

WORK = "/content/E231project"
import shutil, glob
_src = DRIVE_PROJECT_DIR if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "nbtools")) else \
       os.path.dirname(glob.glob(os.path.join(DRIVE_PROJECT_DIR, "*", "nbtools"))[0])
sys.path.insert(0, _src)
from nbtools import cloud, charts, dashboard                 # helpers (Colab side only)
SRC = cloud.prepare_workdir(_src, WORK, PROJECT_ID, BUCKET)  # working copy; your Drive folder is not modified
sys.path.remove(_src); sys.path.insert(0, WORK)
import importlib; [importlib.reload(m) for m in (cloud, charts, dashboard)]
os.chdir(WORK)
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

CONFIG_FILE = "config.yaml"
cfg = yaml.safe_load(open(CONFIG_FILE))
TAG = "full"

cfg["streaming"]["model"] = STREAM_MODEL
cloud.write_yaml(cfg, CONFIG_FILE)
D, I = cfg["data"], cfg["infra"]
R = D["results_prefix"].rstrip("/")
CFG_URI = f"gs://{BUCKET}/configs/{TAG}/{CONFIG_FILE}"
LOCAL = f"/content/results_{TAG}"
DRIVE_OUT = os.path.join(SRC, "results", TAG)
print(f"config {CONFIG_FILE} | cluster {I['cluster_name']} ({I['region']}) | {WORKERS} workers x {I['machine_type']}")
print("results ->", R)

## 3. One-time Google Cloud setup and quota check (safe to rerun)
APIs, roles for the default compute service account, the bucket, and **every quota a cluster uses**: regional CPUs, N2 CPUs, CPUs (all regions), SSD disk and IP addresses. The 4-worker limit you may see comes from these project quotas, not from the scripts. Set a budget alert in Billing yourself.

In [ ]:
cloud.run("gcloud services enable dataproc.googleapis.com compute.googleapis.com storage.googleapis.com cloudresourcemanager.googleapis.com", show_all=True)
num = cloud.out(["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"]).strip()
for role in ["roles/dataproc.worker", "roles/storage.objectAdmin"]:
    cloud.run(["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
               f"--member=serviceAccount:{num}-compute@developer.gserviceaccount.com", f"--role={role}",
               "--condition=None", "--quiet", "--format=none"], show_all=True)
if not cloud.gs_exists(f"gs://{BUCKET}"):
    cloud.run(["gcloud", "storage", "buckets", "create", f"gs://{BUCKET}", f"--location={I['region']}",
               "--uniform-bucket-level-access"], show_all=True)
cloud.run(["gcloud", "storage", "buckets", "update", f"gs://{BUCKET}", "--update-labels=project=da231o"], show_all=True)
print()
MAX_WORKERS = cloud.quota_report(PROJECT_ID, I["region"], cfg, WORKERS)
if MAX_WORKERS < WORKERS:
    print(f"\n>>> WORKERS={WORKERS} does NOT fit. Set WORKERS <= {MAX_WORKERS} in §1 (and rerun §1-§3), or request more "
          "quota: console > IAM & Admin > Quotas & System Limits (not possible on a Free Trial billing account).")

## 4. Raw data into the bucket (several categories at a time; skips files already there)

In [ ]:
cloud.ingest_raw(D["categories"], D["raw_prefix"], RAW_SOURCE, DRIVE_RAW_DIR, parallel=4)

## 5. (Optional) Local correctness test, no cluster

In [ ]:
if RUN_LOCAL_TESTS:
    cloud.run("pip -q install pyspark==3.5.3", show_all=True)
    cloud.run("python tests/test_federated_reference.py 2>/dev/null | grep -E 'PASS|FAIL|ALL'", show_all=True)   # must print ALL PASSED

## 6. Package the code; one cluster for the whole session
Kafka and Spark's Kafka connector are copied into the bucket once (the cluster has no internet access). The cluster is created **only when a stage needs it**, with the Kafka broker on the master when streaming is on.

In [ ]:
if RUN_STREAMING:
    cloud.ensure_kafka_files(cfg)
cluster = cloud.Cluster(cfg, CONFIG_FILE, CFG_URI, WORKERS, WORK, kafka=RUN_STREAMING)
cluster.package()
def done(stage, *names, extra=None):
    """Skip a stage only if its outputs exist AND were produced with the current settings."""
    return cloud.stage_done(cfg, stage, R, [n if n.startswith("gs://") else f"{R}/{n}" for n in names],
                            SKIP_DONE, extra)
P = D["processed_prefix"].rstrip("/")
print("cluster", cluster.name, "exists" if cluster.exists() else "not created yet (created on first use)")

## 7. Module 1 — ETL: CSV → Parquet → clean, deduplicate, filter, per-user temporal split

In [ ]:
if done("etl", f"{P}/train.parquet/_SUCCESS", f"{P}/test.parquet/_SUCCESS", f"{P}/item_category.parquet/_SUCCESS"):
    print("ETL output already in", P, "- skipped (SKIP_DONE)")
else:
    pq = D["raw_parquet_prefix"].rstrip("/")
    todo = [c for c in D["categories"] if not cloud.gs_exists(f"{pq}/category={c}/")]
    print("categories still to convert:", todo or "none")
    cluster.submit("scripts/run_etl.py", "--step=all" if todo else "--step=build",
                   stage="ETL (convert + build)" if todo else "ETL build")
    cloud.mark_done(cfg, "etl", R)
cloud.run(["gcloud", "storage", "ls", P], show_all=True)

## 8. Data profile → data-engineering charts (read-only job)

In [ ]:
if not done("profile", "data_profile.json"):
    cluster.submit("scripts/run_profile.py", stage="profile")
    cloud.mark_done(cfg, "profile", R)
cloud.download_results(R, LOCAL)
charts.data_funnel(LOCAL); charts.data_categories(LOCAL); charts.data_distributions(LOCAL);

## 9. Module 2 — centralized model (bias baseline + Spark ALS on residuals)

In [ ]:
if done("centralized", "centralized_metrics.json", "coldstart_metrics.json", "models/centralized/meta.json"):
    print("centralized results already in the bucket - skipped (SKIP_DONE)")
else:
    if RUN_TUNING:
        cluster.submit("scripts/run_centralized.py", "--tune", stage="tuning")
    cluster.submit("scripts/run_centralized.py", stage="centralized")
    cloud.mark_done(cfg, "centralized", R)
cloud.download_results(R, LOCAL)
charts.accuracy(LOCAL); charts.accuracy_by_category(LOCAL);

## 10. Module 3 — simulated FedAvg (same model; 1 client = control)

In [ ]:
if done("federated", "comparison.json", "federated_convergence.csv", "models/federated/meta.json", extra=FED_CLIENTS):
    print("federated results already in the bucket - skipped (SKIP_DONE)")
else:
    cluster.submit("scripts/run_federated.py", *(["--num-clients", *map(str, FED_CLIENTS)] if FED_CLIENTS else []), stage="federated")
    cloud.mark_done(cfg, "federated", R, extra=FED_CLIENTS)
cloud.download_results(R, LOCAL)
charts.federated(LOCAL);

## 11. Module 4 — scaling benchmarks: strong and weak scaling
One Dataproc job per timed run (fixed executors), as the team designed: strong = fixed training rows, weak = rows grow with executors. The charts show measured speedup and efficiency against ideal, Amdahl's law (strong) and Gustafson's law (weak) fits, and a verdict per curve: **achieved** when efficiency ≥ 0.5 at the largest executor count (a stated rule of thumb; the numbers are printed so you can judge).

In [ ]:
BENCH = {"modes": BENCH_MODES, "executors": BENCH_EXECUTORS, "repetitions": BENCH_REPETITIONS}
if RUN_BENCHMARKS and not done("benchmarks", "scaling_results.csv", extra=BENCH):
    need = max(BENCH_EXECUTORS or cfg["scaling"]["executor_counts"]) // int(I["executors_per_node"])
    assert WORKERS >= need, f"the sweep goes up to {need * int(I['executors_per_node'])} executors: WORKERS must be >= {need}"
    cluster.ensure(); os.makedirs("results", exist_ok=True)
    args = ["--config", CONFIG_FILE]
    if BENCH_MODES: args += ["--modes", *BENCH_MODES]
    if BENCH_EXECUTORS: args += ["--executors", *map(str, BENCH_EXECUTORS)]
    if BENCH_REPETITIONS: args += ["--repetitions", str(BENCH_REPETITIONS)]
    t0 = time.time(); cloud.run(["python", "scripts/run_benchmarks.py", *args]); cloud.STAGE_TIMES.append(("benchmarks", time.time() - t0))
    cloud.mark_done(cfg, "benchmarks", R, extra=BENCH)

cloud.download_results(R, LOCAL)
res = charts.scaling(LOCAL)
if res is not None:
    from IPython.display import display, Image
    display(res[1])                                             # medians, speedup, efficiency per point
    for p in sorted(glob.glob(f"{LOCAL}/*.png")):               # the team's Amdahl / Gustafson plots (plot_scaling.py)
        display(Image(p, width=520))

## 12. Module 4 — Kafka streaming (Navneet's `streaming/`) with a live view
1. Clear the previous run's checkpoint and outputs (a new cluster has an empty Kafka).
2. Start the **scorer** (Structured Streaming, all executors) in the background; it loads the model once into cached serving tables.
3. Start the **producer** (replays test ratings at `streaming.replay_rate_per_sec`) beside it with one small executor.
4. The live view redraws from the per-batch files every 10 s until the scorer stops itself (`idle_stop_seconds` after the last event).

In [ ]:
KJARS = None
if RUN_STREAMING:
    cluster.ensure()
    assert cluster.has_kafka(), (f"cluster {cluster.name} was created without Kafka: run the Delete cell, set RUN_STREAMING=True "
                                 "and rerun from §6")
    KJARS = cloud.kafka_jars(cfg)
    st = cfg["streaming"]
    subprocess.run(f"gcloud storage rm -r {st['checkpoint_location']} {R}/streaming {R}/streaming_latency "
                   f"{R}/streaming_summary.json 2>/dev/null", shell=True)
    SCORER = cluster.submit("scripts/run_streaming.py", "--action=scorer", jars=KJARS, asynchronous=True)
    print(f"waiting {STREAM_WARMUP_SEC}s for the scorer to load the model..."); time.sleep(STREAM_WARMUP_SEC)
    print("scorer:", cluster.job_state(SCORER))
    PRODUCER = cluster.submit("scripts/run_streaming.py", "--action=producer", jars=KJARS, small=True, asynchronous=True)
    t_stream = time.time()
    dashboard.live(cluster, SCORER, R, LOCAL, every_sec=10, watch_job=PRODUCER)
    cloud.STAGE_TIMES.append(("streaming", time.time() - t_stream))
    print("producer:", cluster.job_state(PRODUCER))

## 13. Streaming summary → dashboard (latency, throughput, duplicates, unknown ids)

In [ ]:
if RUN_STREAMING:
    cluster.submit("scripts/run_streaming.py", "--action=summarize", small=True, stage="streaming summary")
    if RUN_RESTART_CHECK:    # same checkpoint: no new batches expected; summary must still report 0 duplicates
        cluster.submit("scripts/run_streaming.py", "--action=scorer", "--max-seconds", "120", jars=KJARS, stage="restart check")
        cluster.submit("scripts/run_streaming.py", "--action=summarize", small=True, stage="streaming summary")
    cloud.download_results(R, LOCAL)
    print(json.dumps(json.load(open(f"{LOCAL}/streaming_summary.json")), indent=2))
    html_path = dashboard.build_html(LOCAL, f"{LOCAL}/streaming_dashboard.html",
                                     title=f"Kafka streaming dashboard ({TAG}, {STREAM_MODEL} model)")
    from IPython.display import HTML, display
    display(HTML(open(html_path).read()))

## 14. All charts in one place, copied to Drive (`results/<tag>/` in the project folder)

In [ ]:
cloud.download_results(R, LOCAL)
figs = charts.all_charts(LOCAL)
shutil.copy(CONFIG_FILE, LOCAL)
shutil.copytree(LOCAL, DRIVE_OUT, dirs_exist_ok=True)
print("job wall times this session:", [(s, round(t)) for s, t in cloud.STAGE_TIMES])
print("saved to Drive:", DRIVE_OUT)

## 15. Delete the cluster (always run this, also after a failure)

In [ ]:
cluster.delete() if "cluster" in globals() else print("no cluster object: nothing was created in this session")